# StudyLens: train the Diagram Scene Graph adapter (A100)

Trains one LoRA adapter for the two model calls the scene graph makes (see `scripts/train_dsg_lora.py`):

* **Grounding**: label names → where each label's leader line ends on the part (`point_2d`) or the labelled region (`bbox_2d`). Supervision: ~25k AI2D train labels.
* **Arrows**: part names → the drawn arrows, by name. Supervision: AI2D train arrows, diagrams without arrows, and synthetic diagrams with random arrow directions.

Everything is **resumable**: checkpoints live on Google Drive; after a disconnect, re-run the cells.

* **GPU:** Runtime → Change runtime type → **A100** (an L4 also works, about 2x slower; a T4 is too slow at 1024 px).
* **Time:** run the probe cell first; it prints seconds per step. Expect roughly 3–5 h for 2 epochs on an A100.
* **Licences:** AI2D (AI2 non-commercial licence, no redistribution) and Qwen2.5-VL-3B (Qwen Research License). The adapter is derived from both: keep the Drive folder private and do not publish the adapter.
* Only the AI2D **train** split is used for training; 150 train diagrams are held out for validation. The AI2D test split is never used here.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/studylens'
!mkdir -p $DRIVE/external $DRIVE/checkpoints
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code and packages

In [ ]:
REPO_URL = 'https://github.com/krish-patel1003/CECS558-term-project-studylens.git'
import os
if not os.path.exists('/content/studylens'):
    !git clone $REPO_URL /content/studylens
%cd /content/studylens
!git pull -q
!pip install -q -r requirements-model.txt
os.environ['STUDYLENS_DEVICE'] = 'cuda'

## 2. AI2D and checkpoints on Drive
The AI2D download requires accepting its licence (`--accept-license`).

In [ ]:
!rm -rf external checkpoints && ln -s $DRIVE/external external && ln -s $DRIVE/checkpoints checkpoints
!python scripts/get_ai2d.py --accept-license
!ls external/ai2d | head

## 3. Probe: memory and speed (3 optimizer steps, nothing saved)

In [ ]:
!python -u scripts/train_dsg_lora.py --probe --run probe

## 4. Train (resumable)
Defaults: 1024 px images (as at inference), LoRA rank 32 on attention + MLP of the language model, lr 1e-4 with cosine decay, 8-example gradient accumulation, 2 epochs, 1,500 synthetic diagrams. If the session disconnects, run this cell again: it resumes from the last saved step (every 100 steps).

In [ ]:
!python -u scripts/train_dsg_lora.py --run dsg_lora --epochs 2

## 5. Validation on held-out AI2D train diagrams
Base model (with the original grounding prompt) vs the adapter: grounding hit rate (same rule as the AI2D evaluation) and arrow precision / recall / F1. Writes `checkpoints/dsg_lora/val_metrics.json`.

In [ ]:
!python -u scripts/train_dsg_lora.py --run dsg_lora --eval --n-eval 80

## 6. Use the adapter
Download `checkpoints/dsg_lora/` from Drive (only `adapter_config.json` and `adapter_model.safetensors` are needed) into the project's `checkpoints/` folder on the laptop, then set in `.env`:

```
QWEN_ADAPTER_PATH=checkpoints/dsg_lora
ADAPTER_KIND=dsg
```

The scene graph then uses the adapter for grounding and arrows; explanations and answers keep the base model.

**Evaluation on the 100 AI2D development diagrams** (system `v3`), here on the GPU: copy the laptop's `external/grounding_runs/` (caches of the baseline answers) and `checkpoints/grouping_model.json` to the same places on Drive first, then run the next cell. It needs PaddleOCR, which is installed in its own environment.

In [ ]:
!python -m venv /content/venv-ocr && /content/venv-ocr/bin/pip install -q -r requirements-ocr.txt
os.environ['OCR_PYTHON'] = '/content/venv-ocr/bin/python'
os.environ['QWEN_ADAPTER_PATH'] = 'checkpoints/dsg_lora'
os.environ['ADAPTER_KIND'] = 'dsg'
!python -u scripts/evaluate_grounding.py --n-parts 60 --n-flow 40 --tag v3
!mkdir -p $DRIVE/results && cp results/grounding_eval_v3.json $DRIVE/results/